# 01_create_materialized_views

Notebook SQL para el modelo dimensional FinPay.


Parámetros:
- `catalog`: catálogo objetivo. Usar `fintech_finpay_dev` para DEV y `fintech_finpay` para PROD.


In [ ]:
--CAMBIAR A fintech_finpay_prod SI SE DESEA DESPLEGAR EN PROD

USE CATALOG fintech_finpay;


In [ ]:
-- 01_create_materialized_views.ipynb
-- Ejecutar una sola vez o ante cambios de esquema.
-- Crea el modelo dimensional sobre la tabla Gold fuente: gold.transactions_enriched.


In [ ]:
CREATE SCHEMA IF NOT EXISTS gold;


In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW gold.fact_transactions
AS
SELECT
    transaction_id,
    date_key,
    transaction_date,
    user_id,
    merchant_id,
    channel_id,
    channel,
    channel_name,
    currency,
    transaction_type,
    status,
    amount,
    transaction_count,
    approved_transaction_count,
    rejected_transaction_count,
    pending_transaction_count,
    reverse_transaction_count,
    CASE
        WHEN transaction_count = 0 THEN 0.0
        ELSE CAST(reverse_transaction_count AS DOUBLE) / CAST(transaction_count AS DOUBLE)
    END AS reverse_rate,
    risk_score,
    is_reverse,
    reference_id,
    _source_file,
    _ingested_at,
    _processed_at,
    _load_date
FROM gold.transactions_enriched
WHERE transaction_id IS NOT NULL;


In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_merchant
AS
SELECT DISTINCT
    merchant_id,
    merchant_name,
    merchant_category AS category,
    merchant_country AS country,
    affiliation_date,
    merchant_status AS affiliation_status,
    risk_level
FROM gold.transactions_enriched
WHERE merchant_id IS NOT NULL;


In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_user
AS
WITH channel_counts AS (
    SELECT
        user_id,
        channel,
        COUNT(*) AS channel_transaction_count
    FROM gold.transactions_enriched
    WHERE user_id IS NOT NULL
      AND channel IS NOT NULL
    GROUP BY user_id, channel
),
preferred_channel AS (
    SELECT
        user_id,
        max_by(channel, channel_transaction_count) AS preferred_channel
    FROM channel_counts
    GROUP BY user_id
),
user_base AS (
    SELECT
        user_id,
        max(full_name) AS full_name,
        max(document_id) AS document_id,
        max(email) AS email,
        max(phone) AS phone,
        max(user_country) AS country,
        max(user_segment) AS risk_segment,
        max(registration_date) AS registration_date
    FROM gold.transactions_enriched
    WHERE user_id IS NOT NULL
    GROUP BY user_id
)
SELECT
    u.user_id,
    u.full_name,
    u.document_id,
    u.email,
    u.phone,
    u.country,
    u.risk_segment,
    p.preferred_channel,
    u.registration_date
FROM user_base u
LEFT JOIN preferred_channel p
    ON u.user_id = p.user_id;


In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_channel
AS
SELECT DISTINCT
    channel_id,
    channel AS channel_code,
    channel_name,
    CASE
        WHEN channel = 'web' THEN 'Digital'
        WHEN channel = 'app' THEN 'Digital'
        WHEN channel = 'pos' THEN 'Physical'
        ELSE 'Unknown'
    END AS channel_group
FROM gold.transactions_enriched
WHERE channel_id IS NOT NULL;


In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_date
AS
SELECT DISTINCT
    date_key,
    transaction_date AS calendar_date,
    day_of_month,
    week_of_year,
    month,
    date_format(transaction_date, 'MMMM') AS month_name,
    quarter,
    year
FROM gold.transactions_enriched
WHERE transaction_date IS NOT NULL;
